Imports

In [ ]:

import torch
import time
from ultralytics import YOLO
from ultralytics.utils.benchmarks import benchmark
import pandas as pd
import sys
from pathlib import Path

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))
    
from config import YAML_PATH, CUSTOM_MODEL_WEIGHTS_PATH

In [ ]:
model = "yolov8n.pt"
epochs = 50

Entrenamiento

In [ ]:
def train_model(model, data_path, epochs):
  model = YOLO(model)
  results = model.train(data=data_path,
                        epochs=epochs,
                        imgsz=640,
                        device='cuda' if torch.cuda.is_available() else 'cpu')

train_model(model, YAML_PATH, epochs)

Evaluacion y comparacion

In [ ]:
import gc

def benchmark_comparativo(model_path, data_path):
    print(f"\n>>> Analizando métricas detalladas: {model_path}")
    
    #Limpiar caché de GPU antes de cada evaluación
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    model = YOLO(model_path)
    
    #Warmup
    dummy_input = torch.randn(1, 3, 640, 640).to("cuda")
    for _ in range(10): _ = model(dummy_input, verbose=False)
    
    metrics = model.val(data=data_path, imgsz=640, verbose=False)
    
    peak_vram = torch.cuda.max_memory_allocated() / 1024**2
    
    return {
        "Modelo": model_path,
        "Precisión (P)": round(metrics.box.mp, 4),
        "Recall (R)": round(metrics.box.mr, 4),
        "F1-Score": round(metrics.box.f1.max(), 4),
        "mAP@50": round(metrics.box.map50, 4),
        "mAP@50-95": round(metrics.box.map, 4),
        "Latencia (ms)": round(metrics.speed['inference'], 2),
        "VRAM (MB)": round(peak_vram, 2)
    }

#Agregar todos los modelos re-entrandos para generar la tabla comparativa
modelos = [CUSTOM_MODEL_WEIGHTS_PATH]
resumen = []

for m in modelos:
    resumen.append(benchmark_comparativo(m, YAML_PATH))

df = pd.DataFrame(resumen)
df.to_csv("comparativa_modelos_alpr.csv", index=False)
print("\n--- TABLA COMPARATIVA ---")
print(df.to_markdown(index=False))